<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/04-agents/03-guardrails-and-budgets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guardrails & Budgets

**Goal:** Add stopping conditions and cost/latency budgets; decide when a pipeline beats an agent.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q groq

In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the GROQ_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except ImportError:
    assert os.environ.get('GROQ_API_KEY'), 'Set GROQ_API_KEY'

from groq import Groq
client = Groq()
MODEL = 'openai/gpt-oss-120b'  # update if you get a 404 — see 00-setup/00-environment.ipynb to list available models

## The production question

The demo agent from notebook 01 works. The production question is different: **what stops it when it doesn't?** An agent loop with no bounds is an unbounded bill — a model that keeps finding one more thing to check, a flaky tool that invites infinite retries, a task that was never completable in the first place. None of these are hypothetical; all of them have produced four-figure API invoices for someone.

This notebook takes the notebook-01 loop and adds five guardrails, one at a time, each with a demonstration:

1. **max_turns** — we had it already; now we'll actually hit it.
2. **Cost budget** — accumulated from real `response.usage` numbers per turn.
3. **Wall-clock timeout.**
4. **Tool-level guards** — a confirmation gate on destructive tools.
5. **Loop detection** — same tool, same args, twice in a row triggers an intervention.

In [ ]:
import json
import re
import time

# Same fake filesystem + calculator as notebook 01, with a few more files
# so multi-step tasks have something to chew on.
FS = {
    "data/q1.txt": "Q1 revenue: $12,400. Churn: 3.1%.",
    "data/q2.txt": "Q2 revenue: $13,900. Churn: 2.8%.",
    "data/q3.txt": "Q3 revenue: $15,200. Churn: 2.9%.",
    "data/q4.txt": "Q4 revenue: $16,800. Churn: 2.5%.",
    "notes/board.md": "Board wants a one-page revenue summary by Friday.",
    "notes/todo.md": "1. Summarize quarters. 2. Flag churn trend.",
}

def list_files():
    return "\n".join(sorted(FS)) if FS else "(no files)"

def read_file(path):
    if path not in FS:
        return f"Error: no file at '{path}'. Call list_files to see what exists."
    return FS[path]

def write_file(path, content):
    FS[path] = content
    return f"Wrote {len(content)} characters to {path}."

def delete_file(path):
    if path not in FS:
        return f"Error: no file at '{path}'."
    del FS[path]
    return f"Deleted {path}."

def calculator(expression):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Error: only numbers and + - * / ( ) are allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

IMPLS = {
    "list_files": list_files, "read_file": read_file, "write_file": write_file,
    "delete_file": delete_file, "calculator": calculator,
}

def tool(name, description, props, required):
    """Build one tool definition in the OpenAI-compatible format Groq uses."""
    return {"type": "function",
            "function": {"name": name, "description": description,
                         "parameters": {"type": "object", "properties": props,
                                        "required": required}}}

_str = lambda: {"type": "string"}
TOOLS = [
    tool("list_files", "List every file path in the workspace, one per line.", {}, []),
    tool("read_file", "Read one file's full contents. Returns text or an error.",
         {"path": _str()}, ["path"]),
    tool("write_file", "Create or overwrite a file. Destructive: replaces existing content.",
         {"path": _str(), "content": _str()}, ["path", "content"]),
    tool("delete_file", "Delete a file permanently. Destructive and irreversible.",
         {"path": _str()}, ["path"]),
    tool("calculator",
         "Evaluate an arithmetic expression using + - * / and parentheses. "
         "Use for all math. Returns the numeric result.",
         {"expression": _str()}, ["expression"]),
]


## Guardrail 1: max_turns — and actually hitting it

Notebook 01's loop already had `max_turns`; it just never fired because the demo task was easy. Here's the compact loop again, returning a structured result instead of a bare string so callers can tell *how* the run ended — a habit every later guardrail builds on. Then we give it more work than the turn budget allows and watch it stop instead of spin.

In [ ]:
def run_agent(task, tools, impls, max_turns=10):
    """Notebook-01 loop, compact, returning a structured result."""
    messages = [{"role": "user", "content": task}]

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages,
        )
        choice = response.choices[0]
        msg = choice.message
        if choice.finish_reason != "tool_calls":
            return {"status": "done", "turns": turn, "text": msg.content or ""}

        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            fn = impls.get(tc.function.name)
            out = str(fn(**args)) if fn else f"Error: unknown tool '{tc.function.name}'."
            print(f"[turn {turn}] {tc.function.name}({tc.function.arguments[:70]}) -> {out.strip()[:70]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

    return {"status": "max_turns", "turns": max_turns,
            "text": f"Stopped at max_turns={max_turns} before finishing."}

# More work than 3 turns allows: six files, each needing its own summary file.
result = run_agent(
    "For every file in the workspace, write a companion file named "
    "<original_path>.summary containing a one-line summary of it. "
    "Do the files one at a time, in alphabetical order.",
    TOOLS, IMPLS, max_turns=3,
)
print("\n", result["status"], "-", result["text"][:120])


Run it and note the result: `status: max_turns` after exactly 3 turns. The task wasn't finished — and that's the point. Without the bound, this run would have continued for however many turns the model needed (fine), but so would a run where the model got stuck re-reading the same file (not fine). The bound converts "potentially unbounded" into "costs at most N calls," and the structured status tells the caller the result is partial rather than silently passing off truncated work as done.

A turn limit is a blunt instrument, though — turns are not dollars. The next guardrail counts what actually costs money.

## Guardrails 2–5: the budgeted loop

Below is the full guarded loop. It's the same ~40 lines with four additions, each marked with a `guard N` comment:

- **Guard 2 (cost budget):** every response carries `response.usage`; we convert tokens to dollars each turn and abort with the partial trace when the spend crosses the budget. This is the guardrail that maps directly to your invoice.
- **Guard 3 (wall-clock timeout):** checked at the top of each turn. Coarse — a single slow API call can overshoot — but it caps drift. Production systems add per-request timeouts underneath.
- **Guard 4 (confirmation gate):** tools listed as destructive don't run unless a `confirm(name, args)` callback approves. Default is **deny** — in a notebook you can pass `confirm=lambda name, args: input(f'{name}? [y/n] ') == 'y'`; in a service this becomes a human-approval queue.
- **Guard 5 (loop detection):** if the model issues the exact same tool call twice in a row, we inject a user-turn intervention telling it to change approach or report failure. Retrying an identical call against a deterministic tool is pure waste.

In [ ]:
# $/MTok illustrative rates (Groq's free tier is $0 -- these let the cost guard do something).
PRICE_IN_PER_MTOK = 0.59
PRICE_OUT_PER_MTOK = 0.79

INTERVENTION = (
    "You already made that exact tool call and got the same result. Do not repeat it. "
    "Try a different approach, or report that you cannot complete the task and why."
)

def run_agent_guarded(task, tools, impls, max_turns=10,
                      max_cost_usd=0.05,        # guard 2
                      max_seconds=120,          # guard 3
                      confirm=None,             # guard 4 (None = deny all destructive calls)
                      destructive=("write_file", "delete_file")):
    messages = [{"role": "user", "content": task}]
    cost, started, last_call, trace = 0.0, time.monotonic(), None, []

    def stopped(status, turn, text=""):
        return {"status": status, "turns": turn, "cost_usd": round(cost, 4),
                "text": text, "partial_trace": trace[-6:]}

    for turn in range(1, max_turns + 1):
        # ---- guard 3: wall-clock timeout ----
        if time.monotonic() - started > max_seconds:
            return stopped("timeout", turn - 1)

        response = client.chat.completions.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages,
        )
        choice = response.choices[0]
        msg = choice.message

        # ---- guard 2: cost budget from real usage numbers ----
        cost += (response.usage.prompt_tokens * PRICE_IN_PER_MTOK
                 + response.usage.completion_tokens * PRICE_OUT_PER_MTOK) / 1_000_000
        if cost > max_cost_usd:
            return stopped("budget_exceeded", turn)

        if choice.finish_reason != "tool_calls":
            return stopped("done", turn, msg.content or "")

        messages.append(msg)
        repeat = False
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)

            # ---- guard 5: same tool + same args as the previous call ----
            sig = (name, json.dumps(args, sort_keys=True))
            repeat = repeat or (sig == last_call)
            last_call = sig

            # ---- guard 4: confirmation gate on destructive tools ----
            if name in destructive:
                approved = confirm(name, args) if confirm else False
                if not approved:
                    out = (f"Denied: {name} requires user approval, which was not "
                           "granted. Do not retry it. Finish the task without this action, "
                           "or report what you would have done instead.")
                    print(f"[turn {turn}] {name} DENIED by confirmation gate")
                    trace.append(f"{name} DENIED")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
                    continue

            fn = impls.get(name)
            out = str(fn(**args)) if fn else f"Error: unknown tool '{name}'."
            print(f"[turn {turn}] {name}({tc.function.arguments[:70]}) -> {out.strip()[:70]}")
            trace.append(f"{name} -> {out.strip()[:60]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

        # ---- guard 5 (continued): inject the intervention as a user turn ----
        if repeat:
            print(f"[turn {turn}] LOOP DETECTED -- injecting intervention")
            messages.append({"role": "user", "content": INTERVENTION})

    return stopped("max_turns", max_turns)


### Demo: guard 2, the cost budget

Give it the same oversized task with a budget that's a fraction of a cent. Run it and note the `budget_exceeded` status, the actual spend, and the `partial_trace` — the caller gets a receipt of what *was* done, not just a failure.

In [ ]:
result = run_agent_guarded(
    "For every file in the workspace, write a companion file named "
    "<original_path>.summary containing a one-line summary of it. "
    "Do the files one at a time.",
    TOOLS, IMPLS,
    max_cost_usd=0.01,   # tiny on purpose -- roughly 2-3 turns at current pricing
)
print()
print(json.dumps({k: v for k, v in result.items() if k != "text"}, indent=2))


One thing the trace makes visible: the per-turn cost *rises* as the run goes on, because every turn re-sends the whole history (notebook 01's closing observation). A fixed per-turn estimate under-predicts long runs; accumulating from `response.usage` is the honest measurement.

### Demo: guard 3, the wall-clock timeout

Same task, generous cost budget, but a timeout shorter than the run. The check happens between turns, so expect the run to finish the in-flight turn and then stop — note the `timeout` status and that `turns` is small.

In [ ]:
result = run_agent_guarded(
    "For every file in the workspace, write a companion file named "
    "<original_path>.summary containing a one-line summary of it. "
    "Do the files one at a time.",
    TOOLS, IMPLS,
    max_cost_usd=1.00,
    max_seconds=8,       # a single turn usually takes a few seconds; two won't fit
)
print()
print(result["status"], "after", result["turns"], "turns,", f"${result['cost_usd']}")


### Demo: guard 4, the confirmation gate

With `confirm=None` every destructive call is auto-denied. The task below *asks* for a write, so run it and watch the model hit the gate, read the denial, and — because the denial message told it what to do — report the summary in its answer instead of retrying the write. (Notebook 02's "errors that teach" rule, applied to a policy message.)

To approve interactively in Colab, pass
`confirm=lambda name, args: input(f"allow {name}? [y/N] ").strip().lower() == "y"`.

In [ ]:
result = run_agent_guarded(
    "Read the four quarterly files in data/ and write a file report.md with total "
    "annual revenue (use the calculator) and one sentence on the churn trend.",
    TOOLS, IMPLS,
    confirm=None,        # auto-deny: write_file and delete_file will be blocked
)
print()
print(result["status"], "-", result["text"][:400])
print("\nreport.md written?", "report.md" in FS)


### Demo: guard 5, loop detection

To trigger a genuine retry loop reliably, we add a rigged tool: `fetch_report` always fails with a *retryable-sounding* error ("temporarily unavailable, try again"). That phrasing is bait — exactly what a flaky upstream API produces — and models will often take it. Run it and watch the sequence: identical call, identical call, `LOOP DETECTED`, intervention injected, and then the model either changes approach or reports failure cleanly instead of burning the remaining turns on retries.

In [ ]:
def fetch_report(name):
    return "Error: report service temporarily unavailable. Try again."

FLAKY_TOOLS = TOOLS + [
    tool("fetch_report",
         "Fetch a named report from the reporting service. Returns the report text.",
         {"name": _str()}, ["name"]),
]
FLAKY_IMPLS = {**IMPLS, "fetch_report": fetch_report}

result = run_agent_guarded(
    "Fetch the 'quarterly' report with fetch_report and summarize it in two sentences.",
    FLAKY_TOOLS, FLAKY_IMPLS,
    max_turns=6,
)
print()
print(result["status"], "-", result["text"][:400])


A note on the mechanism: the intervention goes in as a **user turn**, so it becomes part of the conversation the model conditions on — the same channel a human operator would use. Our detector is deliberately simple (exact same call twice in a row); production versions also catch A→B→A→B alternation and near-duplicate arguments, and they escalate: intervene once, then abort.

## The stop-conditions table

| Condition | What it protects against | What to do on trigger |
|---|---|---|
| `max_turns` | Runaway plans, infinite loops | Fail loud: return status + trace, never silently truncate |
| Cost budget | The unbounded bill; verbose-context blowup | Return partial results + actual spend; alert if it fires often |
| Wall-clock timeout | Hung tools, slow drift, stuck upstreams | Fail loud; pair with per-request timeouts underneath |
| Confirmation gate | Irreversible actions: deletes, sends, payments | Ask a human; deny by default when nobody's there to ask |
| Loop detection | Futile retries against deterministic failures | Intervene once with guidance; abort on the second offense |

The common thread: **every trigger produces a legible outcome** — a status, a spend figure, a partial trace. An agent that fails loudly with a receipt is operable; one that fails silently is a debugging session.

## When a pipeline beats an agent

This is the judgment call interviews probe, and the one this whole section has been building toward. The loop you've now built three times is powerful precisely because the model chooses the next step — and that's also its cost: non-determinism, per-step API spend, and all five guardrails above.

If you can write down the steps ahead of time, **write the pipeline**:

- **Steps known in advance?** Fetch → extract → summarize → store is a pipeline. Code the sequence; call the model inside steps that need language skills.
- **Same shape every run?** If run #1000 should look like run #1, determinism is a feature. Pipelines are testable (assert on each stage), debuggable (logs per stage), and cheap (one model call per model-shaped step, no re-sent history).
- **Does the path genuinely depend on intermediate results?** Only then do you need the loop — debugging ("read the error, decide what to inspect next"), open-ended research, tasks where the file/tool/next question can't be known up front.
- **Unsure?** Start with the pipeline. Promote the one step that genuinely needs branching into a small bounded agent later. The reverse migration — un-agenting a flaky system in production — is much more painful.

Honest observation from the field: a large share of shipped "agents" are pipelines wearing a trench coat — the model "decides" among steps that always run in the same order, and the team pays agent costs (latency, spend, non-determinism, eval difficulty) for pipeline behavior. "We built a pipeline because the workflow was known" is a *stronger* engineering answer than "we built an agent," in interviews and in production.

For the systems view of running these loops at scale — queues, checkpointing, human-in-the-loop approval flows, multi-agent orchestration — see the companion walkthrough: [Agent Orchestration](https://www.calm.rocks/resources/prepare-interview/system-design/agent-orchestration-walkthrough/).

## Exercises

1. **Graceful budget exit.** When the cost budget trips, instead of returning immediately, send *one* final request (no tools, small `max_tokens`) asking the model to summarize progress so far — and add that request's cost to the total. Compare the usefulness of the result against the raw `partial_trace`.
2. **Escalating loop detection.** Extend guard 5 to also catch A→B→A→B alternation (keep the last four signatures), and abort with status `"looping"` if the model repeats *after* an intervention. Verify with the `fetch_report` bait.
3. **Per-tool budgets.** Add a `tool_limits` dict (e.g. `{"fetch_report": 2}`) that denies a tool with a teaching error once its call count is exhausted. Confirm the flaky run now ends with a clean failure report even without the loop detector.
4. **Un-agent it.** Take the guard-4 demo task (read four quarterly files, compute total, report churn trend) and rewrite it as a straight pipeline: one loop over `FS`, one `sum()`, and a *single* model call to phrase the summary. Compare tokens spent and wall-clock time against the agent version, and write one sentence on when the agent version would still be worth it.